# UPAR 2027 Track 1 — train trên Colab

**Chuẩn bị một lần:** upload 2 file vào Google Drive, thư mục `MyDrive/RWS2027/`
- `rws2027_bundle.zip` (682 MB): code + annotations + cache ảnh. **Bắt buộc.**
- `rws2027_ckpt.zip` (348 MB): model DINOv3 đang nộp. Chỉ cần nếu muốn dùng lại.

**Chọn GPU:** Runtime → Change runtime type → GPU (A100 / L4 nhanh nhất, T4 cũng chạy được).

Mọi kết quả train được lưu thẳng vào `MyDrive/RWS2027/runs/`, nên Colab ngắt giữa chừng thì chạy lại cell train là **tự tiếp tục** từ epoch đã lưu (`--resume`).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os; print("CPU cores:", os.cpu_count())

## 2. Mount Drive và giải nén bundle vào ổ local của Colab (nhanh hơn đọc trực tiếp từ Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/RWS2027'
PROJ = '/content/RWS2027'
import os
assert os.path.exists(f'{DRIVE}/rws2027_bundle.zip'), 'Chưa thấy rws2027_bundle.zip trong MyDrive/RWS2027'
if not os.path.exists(f'{PROJ}/cache/train_bytes.npz'):
    !mkdir -p {PROJ} && cd {PROJ} && unzip -q -o {DRIVE}/rws2027_bundle.zip
if os.path.exists(f'{DRIVE}/rws2027_ckpt.zip'):
    !mkdir -p {DRIVE}/runs && cd {DRIVE} && unzip -q -n rws2027_ckpt.zip
# runs/ trỏ sang Drive để checkpoint không mất khi Colab ngắt
!mkdir -p {DRIVE}/runs && rm -rf {PROJ}/runs && ln -s {DRIVE}/runs {PROJ}/runs
!ls {PROJ} {PROJ}/runs

## 3. Cài thư viện (Colab có sẵn torch; timm ghim đúng phiên bản đã test với container chấm bài)

In [ ]:
!pip -q install timm==1.0.30
import torch, timm; print(torch.__version__, timm.__version__, torch.cuda.get_device_name(0))

## 4. Chạy thử nhanh (khoảng 1–2 phút) để chắc mọi thứ ổn

In [ ]:
%cd /content/RWS2027/par
!python train.py --epochs 1 --max-iters 20 --bs 32 --holdout PETA --out /content/smoke 2>&1 | grep --line-buffered -v -i warn

## 5. Giai đoạn 1: 3 model proxy (leave-one-dataset-out)

Mỗi model **không thấy** một dataset khi train, rồi được chấm trên chính dataset đó → mô phỏng test domain lạ.
Thời gian ước lượng trên A100: ~15–20 phút/model (5 epoch). T4: chậm hơn khoảng 3–4 lần.

Chạy lại cell này sau khi bị ngắt → tự bỏ qua model đã xong và tiếp tục model đang dở.

In [ ]:
%cd /content/RWS2027/par
import os
for h in ['PETA', 'Market1501', 'PA100k']:
    out = f'../runs/lodo_base_{h}'
    if os.path.exists(f'{out}/done'):
        print('skip', h); continue
    print(f'===== holdout {h} =====')
    !python -u train.py --holdout {h} --epochs 5 --resume --out {out} 2>&1 | grep --line-buffered -v -i warn | tee -a {out}.log
    if os.path.exists(f'{out}.log') and 'best val' in open(f'{out}.log').read():
        open(f'{out}/done', 'w').close()

## 6. Tóm tắt điểm các epoch (chạy sau khi cell 5 xong, hoặc sau khi bị ngắt)

In [ ]:
!grep -h -E "== ep|best val" /content/RWS2027/runs/lodo_base_*.log

## 7. WiSE-FT trên các model proxy

Trộn trọng số DINOv3 gốc với bản fine-tune theo tỉ lệ α và chấm trên dataset bị giữ lại.
α = 1.0 là model fine-tune nguyên bản. Nếu α < 1 cho điểm cao hơn → nên dùng khi nộp.

In [ ]:
%cd /content/RWS2027/par
for h in ['PETA', 'Market1501', 'PA100k']:
    print('=== holdout', h)
    !python wise.py ../runs/lodo_base_{h} --alphas 0.5 0.6 0.7 0.8 0.9 1.0 2>&1 | grep --line-buffered alpha

## 8. (Sau này) Train model cuối trên train + val

Chỉ chạy khi đã chọn được cấu hình tốt nhất từ proxy.

In [ ]:
%cd /content/RWS2027/par
!python -u train.py --train-splits train val --epochs 10 --resume --out ../runs/final_b16 2>&1 | grep --line-buffered -v -i warn | tee -a ../runs/final_b16.log

## 9. Tải kết quả về máy

Checkpoint nằm trong `MyDrive/RWS2027/runs/`. Đóng gói bài nộp nên làm **trên máy local**
(`par/make_submission.py` + test trong docker), vì cần kiểm tra với container `pytorch:2.4.1`.